# Retrieval Evaluation Across the Dataset

This notebook evaluates retrieval quality across the labelled PubMedQA sample rather than relying on a single qualitative smoke test.

## 1. Evaluation design

Each PubMedQA record provides a question and its associated research context. We use the record's `pubid` as the relevance signal: a retrieved chunk is considered relevant when it comes from the same source document as the query.

This gives us an objective first retrieval benchmark.

## 2. Metrics

**Recall@K** measures the proportion of questions for which the correct source appears in the top K retrieved chunks.

**MRR (Mean Reciprocal Rank)** measures how highly the correct source tends to appear. A higher score means relevant sources tend to appear nearer the top.

## 3. Run the evaluation

In [1]:
import json
import statistics
from pathlib import Path

from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

chunks_path = Path("../data/pubmedqa_chunks.jsonl")
with chunks_path.open() as f:
    chunks = [json.loads(line) for line in f]

questions = {}
for chunk in chunks:
    questions[chunk["pubid"]] = chunk["question"]

model = SentenceTransformer("all-MiniLM-L6-v2")
client = QdrantClient(path="../data/qdrant")

print(f"Source documents evaluated: {len(questions)}")
print(f"Indexed chunks: {len(chunks)}")

/Users/lsolis/Projects/biomedical-rag-evaluation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14309.81it/s]


Source documents evaluated: 500
Indexed chunks: 979


In [2]:
k_values = [1, 3, 5]
hits = {k: 0 for k in k_values}
reciprocal_ranks = []

pubids = list(questions.keys())
embeddings = model.encode(
    [questions[pubid] for pubid in pubids],
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
)

for pubid, query_vector in zip(pubids, embeddings):
    results = client.query_points(
        collection_name="pubmedqa",
        query=query_vector.tolist(),
        limit=max(k_values),
    ).points

    retrieved_pubids = [result.payload["pubid"] for result in results]
    rank = next(
        (i for i, retrieved_pubid in enumerate(retrieved_pubids, start=1)
         if retrieved_pubid == pubid),
        None,
    )

    if rank is None:
        reciprocal_ranks.append(0)
    else:
        reciprocal_ranks.append(1 / rank)
        for k in k_values:
            if rank <= k:
                hits[k] += 1

total = len(pubids)
metrics = {f"Recall@{k}": hits[k] / total for k in k_values}
metrics["MRR"] = statistics.mean(reciprocal_ranks)

for name, value in metrics.items():
    print(f"{name}: {value:.3f}")

client.close()

Batches: 100%|██████████| 16/16 [00:00<00:00, 27.57it/s]


Recall@1: 0.970
Recall@3: 0.988
Recall@5: 0.990
MRR: 0.979


## 4. Interpretation

These metrics provide a baseline for the current embedding model and chunking strategy.

They should not be treated as the final performance of the RAG system. The next useful experiment is to compare retrieval performance under alternative chunking and/or embedding configurations. This turns the project from a simple RAG implementation into an evaluated retrieval system.